### Anrufe

In [1]:
import pandas as pd
import numpy as np

In [2]:
calls_raw = pd.read_excel('Calls (Done).xlsx')
calls = calls_raw.copy()

In [3]:
calls.head(2)

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Dialled Number,Outgoing Call Status,Scheduled in CRM,Tag
0,5805028000000805001,30.06.2023 08:43,John Doe,NaN,Inbound,171.0,Received,NaN,NaN,NaN,NaN
1,5805028000000768006,30.06.2023 08:46,John Doe,NaN,Outbound,28.0,Attended Dialled,NaN,Completed,0.0,NaN


### 2. Bereinigung von Duplikaten und Datentypen

In [4]:
calls = calls.drop(columns=['Dialled Number']) 

In [5]:
# 2. Duplikate entfernen
# Zuerst vollständige Zeilenduplikate, danach nach eindeutiger Call-ID
calls = calls.drop_duplicates()
calls = calls.drop_duplicates(subset=['Id'])

In [6]:
print(f"Verbleibende Zeilen nach Duplikatbereinigung: {len(calls)}")

Verbleibende Zeilen nach Duplikatbereinigung: 95874


### Anruf-Kategorisierung (Tagging)

In [7]:
# 1. Zuerst für alle festlegen'No Answer'
calls['Tag'] = 'No Answer'

# 2. Liste der aktiven Anrufe
success_statuses = ['attended dialled', 'received']

# 3. Maske: Dauer > 0 UND Status einer der zwei "erfolgreichen
mask = (calls['Call Duration (in seconds)'] > 0) & (calls['Call Status'].str.lower().isin(success_statuses))

# 4. Tags für gefilterte Datensätze zuweisen
bins = [0, 15, 300, np.inf]
labels = ['Short Call', 'General Call', 'Long Discussion']

calls.loc[mask, 'Tag'] = pd.cut(calls.loc[mask, 'Call Duration (in seconds)'], 
                                bins=bins, 
                                labels=labels, 
                                include_lowest=False)


In [8]:
print(f"Bereinigung abgeschlossen. Zeilen insgesamt: {len(calls)}")
print(f"Eindeutige IDs: {calls['Id'].nunique()}")
print("\nTag-Statistik:")
print(calls['Tag'].value_counts())

Bereinigung abgeschlossen. Zeilen insgesamt: 95874
Eindeutige IDs: 95874

Tag-Statistik:
Tag
Short Call         34325
General Call       24868
No Answer          22094
Long Discussion    14587
Name: count, dtype: int64


### Fehlwerte auffüllen

In [9]:

calls['Scheduled in CRM'] = calls['Scheduled in CRM'].fillna(0.0)
calls['Outgoing Call Status'] = calls['Outgoing Call Status'].fillna('N/A')
calls['CONTACTID'] = calls['CONTACTID'].fillna('Unknown')

# Prüfung auf Verbleibende Fehlwerte
null_report = calls.isna().sum()
if null_report.sum() == 0:
    print("Perfekt! Die Tabelle „Calls“ enthält keine Leerwerte mehr.")
else:
    print("Verbleibende Fehlwerte in den Spalten:")
    print(null_report[null_report > 0])

Verbleibende Fehlwerte in den Spalten:
Call Duration (in seconds)    83
dtype: int64


In [10]:
# 3. Datentypen konvertieren
# Datumsangaben in das Format datetime für Analysen nach Tagen/Stunden
# Parameter dayfirst=True hinzufügen
calls['Call Start Time'] = pd.to_datetime(calls['Call Start Time'], dayfirst=True)

calls['Scheduled in CRM'] = calls['Scheduled in CRM'].fillna(0).astype(int)
calls['Outgoing Call Status'] = calls['Outgoing Call Status'].fillna('Inbound & Missed Calls').astype('category')

# 'Call Type', 'Call Status', 'Call Owner Name' in kategorialen Datentyp umwandeln
calls[['Call Type', 'Call Status', 'Call Owner Name' ]] = calls[['Call Type', 'Call Status', 'Call Owner Name' ]].astype('category')

# Dauer in numerischen Typ konvertieren (Text wird zu NaN, anschließend Ersetzung durch 0)
calls['Call Duration (in seconds)'] = pd.to_numeric(calls['Call Duration (in seconds)'], errors='coerce').fillna(0)

In [11]:
contacts_raw = pd.read_excel('Contacts (Done).xlsx')
contacts = contacts_raw.copy()

In [ ]:
#Erstellung einer neuen Datei

In [131]:
calls.to_csv('Calls_Processed_Final.csv', index=False)